# 📘 Annotated edition · 05_methods

A copy of `notebooks/05_methods.ipynb`. Original cells and saved outputs are unchanged. New **📘 Explanation** and **✅ Answer** cells are added.

**The point of this notebook:** five analysis methods, one per brief, each written **vectorised** (whole-column operations, no Python loops over rows) so they run on the full pack in seconds:

| § | Method | Brief | Question it answers |
|---|---|---|---|
| 1 | **The tree** | A | Tests grew 5%: how much from more people, more visits per person, more tests per visit? |
| 2 | **The ladder** | B | Chennai and Jaipur look down: is it real, or a system or mix artefact? |
| 3 | **The bridge** | C | From invoiced to collected: where did the money go? |
| 4 | **The chance reference** | D | Is one clinic's no-show rate bad, or just noise? |
| 5 | **The fair comparison** | E | Did the campaign raise bookings, or was it season and selection? |

**About outputs:** in the saved file only the first cells (to §1) have outputs. Cells from §2 on were never run. For those, the **📘 Reading the output** notes quote what the cells produce, recomputed on the same clean data with the notebook's own filters and seed.

⚠️ **Carried-over issue:** the half-year filter uses `between(...)` with date-only bounds, so **30 June is excluded** (see notebook 04). Every level in this notebook is about 0.5% low. The rates barely move.

> ℹ️ **One line changed in this annotated copy:** `ROOT = ... if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else ...`. The original only recognised a folder called `notebooks`. Run from `notebook_new/`, it would have looked for `notebook_new/data/raw` and failed. Nothing else in the code was changed.

# 05 · The methods: tree, ladder, bridge, chance reference, fair comparison

**Produces:** the numbers behind your claim, each with its denominator and period, and the chart for slide 4.

**Needs:** notebook 03 run.

Five methods from Weeks 1 and 2, written vectorised so they run on the whole pack in seconds. Use the one your brief needs; read the others so you can follow the other groups on presentation day.

#### 📘 Explanation
"Use the one your brief needs; read the others." On presentation day other groups present the other methods, and the panel expects you to follow and question them.

## CONFIG

#### 📘 Explanation: the CONFIG cell, line by line

This cell is the same in notebooks 01–06. Explained once in full:

| Code | What it does and why |
|---|---|
| `import sys, pathlib, time` | `sys` to edit the import path, `pathlib` for file paths that work on Windows and Linux, `time` to time steps. |
| `ROOT = pathlib.Path.cwd().parent if ... == "notebooks" else pathlib.Path.cwd()` | Finds the **project root**. Jupyter usually starts inside the notebook's folder, so the root is one level up. When run from the root itself (for example a script), `cwd()` is already the root. Every path below is built from `ROOT`, so the notebook works on any machine. |
| `sys.path.insert(0, str(ROOT / "src"))` | Puts `src/` at the front of Python's import search path, so `from kalpa import ...` finds the project's own package without `pip install`. |
| `import numpy as np` / `import pandas as pd` | The two core libraries: arrays and fast maths / tables. |
| `from kalpa import loaders, models, checks, logs` | The shared project package: **loaders** read the raw files, **models** are pydantic "contracts" for a valid record, **checks** print PASS/FAIL, **logs** append rows to the markdown logs in `docs/`. |
| `pd.set_option("display.max_columns", 40)` / `("display.width", 160)` | Print wide tables in full instead of truncating them with `...`. |
| `CONFIG = {...}` | All run-specific settings in one place (see notebook 00): brief, group, `raw` and `clean` folders as `pathlib` paths, the two half-years being compared, the seed, and `workers` (`None` means use every CPU core in the process pool). |
| `CONFIG["clean"].mkdir(parents=True, exist_ok=True)` | Creates `data/clean/` if it doesn't exist, and doesn't fail if it does. A fresh clone can run straight away. |
| `rng = np.random.default_rng(CONFIG["seed"])` | A seeded random generator **object**. Anything random (the shuffle test in 05) uses `rng`, so results are reproducible. |
| `print(pd.__version__, np.__version__)` | Prints library versions (**2.3.3** and **2.3.5** in the saved run). If a re-run elsewhere gives different numbers, a version difference is the first suspect. pandas 3.x, for example, changed the default string dtype. |


**Extra lines in this notebook (loading and labelling the halves):**
- `bk = pd.read_csv(... "bookings_clean.csv.gz", parse_dates=["slot_at","collected_at"], dtype=...)` reads the clean booking-grain table from 03.
- `bk["half"] = np.where(between(period_last), "H1_2025", np.where(between(period_this), "H1_2026", "other"))`: nested `np.where` is a vectorised if/elif/else that labels each booking. ⚠️ `between` with `'2025-06-30'` stops at 00:00 on 30 June.
- `bk = bk[bk["half"] != "other"]` keeps only the two halves being compared.
- `groupby("half").agg(bookings=("booking_id","size"), tests=("n_tests","sum"), people=("person_id","nunique"))`: the three base counts. `size` counts rows (one per booking), `sum` adds tests, `nunique` counts distinct **people** (after the de-duplication in 03, not patient ids).

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at", "collected_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
bk["half"] = np.where(bk["slot_at"].between(*CONFIG["period_last"]), "H1_2025", np.where(bk["slot_at"].between(*CONFIG["period_this"]), "H1_2026", "other"))
bk = bk[bk["half"] != "other"]
print(bk.groupby("half").agg(bookings=("booking_id", "size"), tests=("n_tests", "sum"), people=("person_id", "nunique")))

2.3.3 2.3.5
         bookings   tests  people
half                             
H1_2025    187771  431261   93926
H1_2026    193005  452933   97501


#### 📘 Reading the output
| | bookings | tests | people |
|---|---|---|---|
| H1 2025 | 187,771 | 431,261 | 93,926 |
| H1 2026 | 193,005 | 452,933 | 97,501 |

With full days (SQL in 04) the figures are 188,803 / 433,611 / 94,275 and 194,020 / 455,383 / 97,827.

## 1 · The tree (brief A)

Tests = people × visits per person × tests per visit. Growth in a product is close to the sum of growths in the factors (exactly so in logs). Compute each factor for both halves and check that they multiply back.

#### 📘 Explanation: the tree

**Tests = people × (visits ÷ people) × (tests ÷ visits).** The middle terms cancel, so this is an identity, always true.

Growth of a product: (1+g) = (1+g₁)(1+g₂)(1+g₃). Percentages **don't add exactly** (5% ≠ 3.8% − 1.0% + 2.2% exactly, because of cross terms), but **logs do**: ln(1+g) = ln(1+g₁) + ln(1+g₂) + ln(1+g₃). Log growths are the exact way to say "how much of the growth came from each factor".

#### 📘 Explanation: the `tree` function and the log check

- `tree(df, by=None)` groups by `half` (and optionally another column such as city) and computes `tests` (sum of n_tests), `visits` (number of bookings) and `people` (distinct persons), then the two ratios `visits_per_person` and `tests_per_visit`.
- `([by] if by else []) + ["half"]` builds the list of grouping columns.
- `t.xs("H1_2026") / t.xs("H1_2025") - 1`: `.xs` takes the cross-section (the row) for one half. Dividing row by row gives growth for every column at once, then `* 100` for percent.
- `logs_ = np.log(t.xs("H1_2026") / t.xs("H1_2025"))`: log growth per column.
- The check: `ln(people) + ln(visits/person) + ln(tests/visit) == ln(tests)` to within 10⁻⁹. It must pass by algebra, so a FAIL would mean a bug (for example visits counted differently in two places).

In [2]:
def tree(df, by=None):
    g = df.groupby(([by] if by else []) + ["half"], observed=True).agg(tests=("n_tests", "sum"), visits=("booking_id", "size"), people=("person_id", "nunique"))
    g["visits_per_person"] = g["visits"] / g["people"]
    g["tests_per_visit"] = g["tests"] / g["visits"]
    return g

t = tree(bk)
growth = (t.xs("H1_2026") / t.xs("H1_2025") - 1) * 100
print(growth.round(2))
# log-decomposition: the branch growths sum to the total growth exactly
logs_ = np.log(t.xs("H1_2026") / t.xs("H1_2025"))
checks.check("branches multiply back to the total (log decomposition)", abs(logs_["people"] + logs_["visits_per_person"] + logs_["tests_per_visit"] - logs_["tests"]) < 1e-9,
             f"people {logs_['people']:+.4f} + visits/person {logs_['visits_per_person']:+.4f} + tests/visit {logs_['tests_per_visit']:+.4f} = {logs_['tests']:+.4f}")

tests                5.03
visits               2.79
people               3.81
visits_per_person   -0.98
tests_per_visit      2.18
dtype: float64
PASS  branches multiply back to the total (log decomposition)   people +0.0374 + visits/person -0.0099 + tests/visit +0.0215 = +0.0490


True

#### 📘 Reading the output

| Factor | Growth % | Log growth |
|---|---|---|
| **tests** | **+5.03** | +0.0490 |
| people | +3.81 | +0.0374 |
| visits per person | −0.98 | −0.0099 |
| tests per visit | +2.18 | +0.0215 |

(visits +2.79% = people × visits per person.)

**PASS:** 0.0374 − 0.0099 + 0.0215 = 0.0490. **In words:** growth came from **more people** (about three quarters of it) and **more tests per visit** (about 44%, partly through packages). People came **slightly less often** (−1%, offsetting about 20%). Shares of log growth: 0.0374/0.0490 = 76%, 0.0215/0.0490 = 44%, −0.0099/0.0490 = −20%.

#### 📘 Explanation: the tree by city

`tree(bk, "city")` gives the same factors per city. `.xs("H1_2026", level="half")` picks one half from the two-level index (city, half). Dividing two city-indexed tables aligns on city automatically. Sorted by test growth.

In [3]:
by_city = tree(bk, "city")
city_growth = ((by_city.xs("H1_2026", level="half") / by_city.xs("H1_2025", level="half") - 1) * 100).round(1)
city_growth[["tests", "people", "visits_per_person", "tests_per_visit"]].sort_values("tests")

,tests,people,visits_per_person,tests_per_visit
city,,,,
Jaipur,-10.6,-12.0,-0.1,1.8
Chennai,0.9,-4.8,3.1,2.8
Hyderabad,7.2,8.1,-2.5,1.8
Bengaluru,7.6,7.2,-2.3,2.8
Pune,8.3,8.9,-2.2,1.6
Ahmedabad,10.5,9.7,-1.1,1.8


#### 📘 Reading the output

| City | Tests | People | Visits per person | Tests per visit |
|---|---|---|---|---|
| **Jaipur** | **−10.6** | **−12.0** | −0.1 | +1.8 |
| Chennai | +0.9 | **−4.8** | +3.1 | +2.8 |
| Hyderabad | +7.2 | +8.1 | −2.5 | +1.8 |
| Bengaluru | +7.6 | +7.2 | −2.3 | +2.8 |
| Pune | +8.3 | +8.9 | −2.2 | +1.6 |
| Ahmedabad | +10.5 | +9.7 | −1.1 | +1.8 |

The network's +5% hides two very different stories. Four cities grow 7–10% on **new people**. **Jaipur and Chennai lose people**, and they're exactly the two cities that moved to nova in April 2026. That's the ladder's question (§2): is it the business, or the system change? Tests per visit rise *everywhere* (+1.6 to +2.8), which looks like a network-wide effect (pricing, packages, ordering habits), not a city one.

> **Interview angle.** Revenue grew 10 percent and patients grew 4. Decompose the rest without a calculator, then say what you would check before presenting it.
>
> *What a good answer covers:* Roughly 6 points sit in visits per patient and revenue per visit together; log growth makes the split exact. Check that the patient count is a count of people and not of ids (duplicates), that both periods use the same definition of a visit, and that a system change did not move the grain.

#### ✅ Answer: revenue grew 10% and patients 4%. Decompose the rest without a calculator, then say what you'd check before presenting it

**Decomposition:** Revenue = patients × visits per patient × revenue per visit. In log terms, ln(1.10) ≈ 0.095 and ln(1.04) ≈ 0.039, so the rest is ≈ 0.056, roughly **5.8%**, or about **6 points** to split between **visits per patient** and **revenue per visit** together. (Quick approximation: 10 − 4 = 6. The exact figure is 1.10 / 1.04 = 1.0577, i.e. +5.8%.) To split those 6 points, compute visits per patient for both periods. If it's flat, almost all of it is revenue per visit: price, mix, more tests per visit, packages, home-collection fees.

**What I'd check before presenting it:**
1. **Patients are people, not ids.** Duplicate registrations inflate "patients". In this pack, 309,217 ids are 303,153 people. A change in duplicate rates between years would fake patient growth.
2. **Same definition of a visit in both periods.** Here a system change (legacy → nova) could count a reschedule as two visits. That's why the identity rule comes first.
3. **Same grain.** Revenue summed at booking-test grain vs booking grain double-counts the home fee.
4. **Same period length and boundaries** (the 30 June bug), and the same cities in both periods.
5. **Revenue definition**: invoiced or collected, with or without refunds, list price or actual price.
6. **Whether one segment drives it.** Run the tree by city. Here two cities shrink while four grow, so the total hides that.

## 2 · The ladder (brief B)

Confirm the drop, compare like with like, decompose, isolate, hypothesise. In code the ladder is a sequence of groupbys where each step changes one thing: the grain, the window, the segment.

#### 📘 Explanation: the ladder

Five rungs, each changing **one thing** so you know what caused any difference:
1. **Confirm** the drop exists in the clean data (not only in a raw report).
2. **Like with like**: same quarter last year, same grain (bookings, after the identity rule), same cities.
3. **Decompose**: by system, channel, clinic, test type.
4. **Isolate**: find where the change is concentrated.
5. **Hypothesise**: only now propose a cause, and say what evidence would confirm it.

#### 📘 Explanation: rung 1–3, quarters by city and system

- `bk["quarter"] = bk["slot_at"].dt.to_period("Q").astype(str)` gives labels like `"2026Q2"`.
- Filter to Chennai and Jaipur, group by `(city, quarter, system)`, count bookings and sum tests.
- `pivot_table(index=["city","quarter"], columns="system", values=["bookings","tests"], aggfunc="sum", fill_value=0)` puts legacy and nova **side by side**, with 0 where a system has no data. Because `bk` contains only the two halves, the quarters shown are 2025Q1, 2025Q2, 2026Q1 and 2026Q2.

In [ ]:
bk["quarter"] = bk["slot_at"].dt.to_period("Q").astype(str)
q = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter", "system"], observed=True).agg(bookings=("booking_id", "size"), tests=("n_tests", "sum")).reset_index()
pivot = q.pivot_table(index=["city", "quarter"], columns="system", values=["bookings", "tests"], aggfunc="sum", fill_value=0)
pivot

#### 📘 Reading the output (cell not run in the saved file. Values recomputed)

| City | Quarter | Bookings (legacy / nova) | Tests (legacy / nova) |
|---|---|---|---|
| Chennai | 2025Q1 | 17,363 / 0 | 39,950 / 0 |
| Chennai | 2025Q2 | 17,221 / 0 | 39,393 / 0 |
| Chennai | 2026Q1 | 17,545 / 0 | 41,476 / 0 |
| Chennai | 2026Q2 | 0 / **16,408** | 0 / 38,579 |
| Jaipur | 2025Q1 | 10,387 / 0 | 23,869 / 0 |
| Jaipur | 2025Q2 | 9,025 / 0 | 20,779 / 0 |
| Jaipur | 2026Q1 | 8,750 / 0 | 20,369 / 0 |
| Jaipur | 2026Q2 | 0 / **8,309** | 0 / 19,556 |

The switch is clean: legacy ends in Q1 and nova begins in Q2, with no overlap month to calibrate against. **Chennai** was flat or up through Q1 2026 (+1.0% on Q1 2025), then fell in the first nova quarter. **Jaipur** was *already* declining on legacy: Q1 2026 was 15.8% below Q1 2025, before the switch. Jaipur's drop is not a migration artefact. Chennai's might be.

#### 📘 Explanation: rung 2, like with like

- `yoy`: bookings per city per quarter, with `unstack("quarter")` turning quarters into columns.
- `pct_change(table, later, earlier, label)` prints `later / earlier − 1` in %, **only if both periods exist**. If one is missing (for example the small sample pack), it says so instead of failing or printing a meaningless number. "Say so rather than fail quietly."
- Called for Q2 2026 vs Q2 2025: the same quarter a year apart, so seasonality cancels.

In [ ]:
# Rung 2, like with like: the same quarter a year earlier, per city, at booking grain, after the identity rule
yoy = bk[bk["city"].isin(["Chennai", "Jaipur"])].groupby(["city", "quarter"], observed=True)["booking_id"].size().unstack("quarter")
def pct_change(table, later, earlier, label):
    """Like with like means both periods must exist; say so when one does not, rather than fail quietly."""
    if later in table and earlier in table:
        print(((table[later] / table[earlier] - 1) * 100).round(1).rename(label))
    else:
        print(f"{label}: period {earlier if earlier not in table else later} is not in this pack (sample data?); nothing to compare yet")
pct_change(yoy, "2026Q2", "2025Q2", "Q2 2026 vs Q2 2025, bookings, %")

#### 📘 Reading the output (recomputed)
**Q2 2026 vs Q2 2025, bookings: Chennai −4.7%, Jaipur −7.9%.** Both cities are down year on year at booking grain, *after* the identity rule removed nova's double-counted reschedules. Next rungs: split Q2 by channel and clinic, and check the "cancelled_or_missing" rate (the reschedule–collection mismatch in 03 makes nova look worse on collection rates, though not on booking counts).

## 3 · The bridge (brief C)

From invoiced to collected, every bar a count and an amount, and the two sides reconcile.

#### 📘 Explanation: the bridge

Start with **invoiced** (what we billed in Q2 2026), end with **collected in the quarter**, and explain the gap with bars. Each bar has a **count and an amount**, and start + bars must equal end exactly. That's the "reconcile" check.

#### 📘 Explanation: building the bridge

1. Load clean invoices and payments. `q2` = invoices dated 1 Apr – 30 Jun 2026 (`invoice_date` has no time, so `between` is safe here).
2. `cap` = captured payments only. `cutoff` = 30 June 23:59:59.
3. `cap.groupby("invoice_id").agg(paid_rs=sum, n_pay=size, first_paid=min)`: **aggregate payments to one row per invoice first**, so the join can't fan out (the 1,925 multi-payment invoices from 03).
4. `q2.merge(..., how="left")`: every Q2 invoice, with its payment summary or NaN.
5. Classify:
   - `unpaid`: no captured payment at all;
   - `paid_after_cutoff`: first payment after the quarter closed (**timing**);
   - otherwise collected in the quarter, possibly a different amount.
6. `steps`: the bars, negative for money that didn't arrive in the quarter. The amount-difference bar is `paid − invoiced` for the in-quarter group (positive = overpaid).
7. `checks.bridge_balances(start, steps, end)` prints each bar and checks `start + Σ steps == end` within ₹0.50.
8. `orphans`: captured payments whose `invoice_id` is in **no** invoice (the anti-join), which sit outside the bridge.

In [ ]:
inv = pd.read_csv(CONFIG["clean"] / "invoices_clean.csv.gz", parse_dates=["invoice_date"], dtype={"invoice_id": "string"})
pay = pd.read_csv(CONFIG["clean"] / "payments_clean.csv.gz", parse_dates=["created_at"], dtype={"invoice_id": "string"})
q2 = inv[inv["invoice_date"].between("2026-04-01", "2026-06-30")]
cap = pay[pay["status"] == "captured"]
cutoff = pd.Timestamp("2026-06-30 23:59:59")
m = q2.merge(cap.groupby("invoice_id").agg(paid_rs=("amount_rs", "sum"), n_pay=("payment_id", "size"), first_paid=("created_at", "min")), on="invoice_id", how="left")
unpaid = m["paid_rs"].isna()
paid_after_cutoff = m["first_paid"] > cutoff
steps = {
    "invoices never paid": -m.loc[unpaid, "amount_rs"].sum(),
    "paid after the quarter closed (timing)": -m.loc[~unpaid & paid_after_cutoff, "amount_rs"].sum(),
    "over- or under-paid (amount differences)": (m.loc[~unpaid & ~paid_after_cutoff, "paid_rs"] - m.loc[~unpaid & ~paid_after_cutoff, "amount_rs"]).sum(),
}
collected_in_q = m.loc[~unpaid & ~paid_after_cutoff, "paid_rs"].sum()
checks.bridge_balances("Q2 2026 invoiced to collected", q2["amount_rs"].sum(), steps, collected_in_q)
orphans = cap[~cap["invoice_id"].isin(inv["invoice_id"])]
print(f"captured payments matching no invoice: {len(orphans):,} rows, Rs {orphans['amount_rs'].sum():,.0f}")

#### 📘 Reading the output (recomputed)

| Bar | Invoices | Amount (₹) |
|---|---|---|
| **Invoiced, Q2 2026** | 79,246 | **132,847,274** |
| − never paid | 6,581 | −11,107,904 |
| − paid after the quarter closed (timing) | 1,763 | −2,982,123 |
| ± over- or under-paid (amount differences) | 335 (324 over, 11 under) | +529,126 |
| **= Collected in the quarter** | 70,902 | **119,286,373** (89.8% of invoiced) |

**PASS: the bridge balances** (difference ₹0.00).
Plus, outside the bridge: **4,588 captured payments (₹7,269,408) that match no invoice at all**, referencing an `INV-2025-9xxxxx` range billing never issued.

Interpretation: of the ₹13.56 M shortfall, ₹2.98 M is **timing** (it arrives in Q3), ₹11.11 M is **unpaid invoices** (the possible leak, which needs ageing: how old are they?), and the amount bar is *positive*, since overpayments (likely double charges) exceed underpayments. Those overpayments are owed back to patients.

> **Interview angle.** Finance says collected is 4 percent below invoiced. Before you call it a leak, what three things do you separate it into?
>
> *What a good answer covers:* Timing (invoiced in the period, paid after the cutoff), unmatched (payments whose reference does not resolve to an invoice, and invoices with no payment at all), and amount differences (fees, partial payments, retries). Each is a bar with a count and an amount, and only the middle one is a possible leak.

#### ✅ Answer: finance says collected is 4% below invoiced. Before calling it a leak, which three things do you separate it into?

1. **Timing.** Invoiced in the period but paid **after the cutoff**. Not lost, just late. In our Q2 2026 bridge: 1,763 invoices, ₹2.98 M. Check: does it shrink when you re-run a month later?
2. **Unmatched.** Money and invoices that exist but **don't link**: payments whose reference doesn't resolve to an invoice (4,588 captured payments, ₹7.27 M) and invoices with no payment found (6,581, ₹11.11 M in Q2). Before calling these unpaid, try fuzzy matching (amount + date + patient) on the residue. Some "unpaid" invoices may have been paid with a mistyped reference, which would show up exactly among the orphan payments.
3. **Amount differences.** Matched, but **a different amount**: gateway fees, partial payments, discounts, retries captured twice (324 overpaid invoices, +₹541 k), underpayments (11, −₹12 k).

Each is a **bar with a count and an amount**, and the bars reconcile to the total. **Only the middle one is a possible leak**, and only after matching is exhausted. Timing resolves itself. Amount differences are mostly explainable (fees, refunds owed).

## 4 · The chance reference (brief D)

How often would a clinic of this size look this bad by chance alone? Shuffle the clinic labels across bookings many times with NumPy, recompute the clinic's rate each time, and see where the real rate falls. Vectorised: one 2-D array of permutations, no Python loop over shuffles.

#### 📘 Explanation: the chance reference (a permutation test)

Question: one clinic has the highest no-show rate. Is that a **real** difference, or would *some* clinic of that size look that bad by luck alone?

The method: pretend clinic labels mean nothing (the **null hypothesis**). Randomly assign the clinic's number of bookings from the whole network, compute the no-show rate, repeat thousands of times. That gives the range of rates **chance alone** produces. If the real rate is far outside it, chance is not a good explanation.

#### 📘 Explanation: no-show rates per clinic

- `d = bk[bk["channel"] == "clinic"]`: clinic visits only (home visits have no clinic waiting room).
- `d["no_show"] = (d["status"] == "no_show").astype(int)` gives 1 for a no-show, 0 otherwise. The mean of a 0/1 column is a **rate**.
- `groupby("clinic_id")["no_show"].agg(["mean","size"])`: rate and number of bookings per clinic.
- `network`: the overall rate. `diff_pts`: clinic minus network, in percentage points.

⚠️ **Denominator choice:** here `status == "cancelled_or_missing"` counts as 0 (not a no-show), so cancellations are in the denominator. SQL query 7 in notebook 04 only uses collected + no-show bookings. The two rates differ (0.164 vs about 0.170). Pick one definition and state it.

In [ ]:
d = bk[bk["channel"] == "clinic"].copy()
d["no_show"] = (d["status"] == "no_show").astype(int)
rates = d.groupby("clinic_id", observed=True)["no_show"].agg(["mean", "size"]).rename(columns={"mean": "rate", "size": "bookings"})
network = d["no_show"].mean()
rates["diff_pts"] = (rates["rate"] - network) * 100
print(f"network no-show rate {network:.3f}")
print(rates.sort_values("rate", ascending=False).head(5).round(3))

#### 📘 Reading the output (recomputed)

Network no-show rate **0.164** (16.4% of clinic bookings in the two halves).

| Clinic | Rate | Bookings | vs network |
|---|---|---|---|
| **CHE-03** | **0.190** | 7,411 | **+2.6 pts** |
| PUN-04 | 0.172 | 4,573 | +0.8 |
| HYD-03 | 0.171 | 7,072 | +0.6 |
| AHM-05 | 0.170 | 4,779 | +0.6 |
| AHM-02 | 0.169 | 9,558 | +0.4 |

CHE-03 stands out clearly. Every other clinic is within about 1 point of the network.

#### 📘 Explanation: the shuffle, slow and fast

**`shuffle_reference_loop`**: the literal method. `rng.choice(len(outcomes), n_clinic, replace=False)` picks n random bookings from the network **without replacement**, takes the mean of their 0/1 outcomes, and repeats `n_shuffles` times in a Python loop. Each draw permutes ~230,000 clinic-booking indices, so it's slow (only 200 shuffles are run).

**`shuffle_reference`**: the same null hypothesis, in one call. When you draw `n_clinic` items without replacement from a population with `ngood` ones and `nbad` zeros, the number of ones drawn follows the **hypergeometric distribution**. `rng.hypergeometric(ngood, nbad, n_clinic, size=20000)` draws 20,000 such counts at once. Dividing by `n_clinic` gives 20,000 shuffled rates. That's mathematically the same distribution as the loop, and roughly a thousand times faster.

Then:
- `worst = rates["rate"].idxmax()`: the clinic with the highest rate (CHE-03).
- The two methods are timed and their means compared. They should agree.
- `p = (ref >= real).mean()`: the share of shuffled worlds at least as bad as the real one (a one-sided p-value).
- `np.percentile(ref, 2.5)` and `97.5` give the middle 95% of chance outcomes.

Note: the seeded `rng` from CONFIG is used for both, so re-runs give the same numbers.

In [ ]:
def shuffle_reference_loop(outcomes: np.ndarray, n_clinic: int, n_shuffles: int, rng) -> np.ndarray:
    """The literal shuffle: relabel n_clinic random bookings as 'this clinic', take their rate, repeat. Clear, and slow."""
    return np.array([outcomes[rng.choice(len(outcomes), n_clinic, replace=False)].mean() for _ in range(n_shuffles)])

def shuffle_reference(outcomes: np.ndarray, n_clinic: int, n_shuffles: int, rng) -> np.ndarray:
    """The same null, vectorised. For a 0/1 outcome, drawing n_clinic bookings without replacement from the network and
    counting no-shows is a hypergeometric draw, which NumPy produces in one call. Same distribution as the loop, a thousand times faster."""
    ngood = int(outcomes.sum()); nbad = int(len(outcomes) - ngood)
    return rng.hypergeometric(ngood, nbad, n_clinic, size=n_shuffles) / n_clinic

worst = rates["rate"].idxmax()
y = d["no_show"].to_numpy()
t0 = time.time(); ref_loop = shuffle_reference_loop(y, int(rates.loc[worst, "bookings"]), 200, rng); t_loop = time.time() - t0
t0 = time.time(); ref = shuffle_reference(y, int(rates.loc[worst, "bookings"]), 20000, rng); t_vec = time.time() - t0
print(f"loop: 200 shuffles in {t_loop:.2f}s, mean {ref_loop.mean():.4f}; vectorised: 20,000 draws in {t_vec:.3f}s, mean {ref.mean():.4f}")
real = rates.loc[worst, "rate"]
p = float((ref >= real).mean())
print(f"{worst}: real rate {real:.3f} on {int(rates.loc[worst, 'bookings']):,} bookings; shuffled rates mean {ref.mean():.3f}, 95% band {np.percentile(ref, 2.5):.3f} to {np.percentile(ref, 97.5):.3f}; share of shuffles at least as extreme: {p:.4f}")

#### 📘 Reading the output (recomputed with the same seed)

- loop: 200 shuffles, mean **0.1647**. Vectorised: 20,000 draws, mean **0.1642**, in milliseconds. The means agree (both ≈ the network rate, as they must under the null).
- **CHE-03: real rate 0.190 on 7,411 bookings. Chance band (95%) 0.156 to 0.173. Share of shuffles at least as extreme: 0.0000** (none of 20,000).

So CHE-03's rate is **not** plausibly chance, given random allocation of bookings. But that assumption is exactly what the next cell tests: bookings are *not* randomly allocated, because clinics differ in when and how people book.

#### 📘 Explanation: like with like, adjusting for mix

No-shows depend heavily on **when** (evening slots are missed more) and **how** the booking was made (phone bookings are missed more than app bookings). A clinic with many evening phone bookings will look bad even if it's run perfectly.

- `pd.cut(hour, [0, 10, 14, 24], labels=["morning","midday","evening"])`: buckets the slot hour into (0,10], (10,14], (14,24]. Slots start at 06:00, so no hour 0 is lost.
- `mix_rate`: the **network** no-show rate for each (slot bucket, source) combination.
- `d.merge(mix_rate, on=[...], how="left", validate="many_to_one")` gives each booking the network rate for *its* slot and source, its "expected" rate.
- Per clinic: actual rate, the **mean expected rate given its own mix**, and `excess_after_mix_pts` = actual − expected. (This is *indirect standardisation*.)

In [ ]:
# Like with like: the same slot, the same source. A rate that survives this comparison is about the clinic; one that does not is about its mix.
d["slot_bucket"] = pd.cut(d["slot_at"].dt.hour, [0, 10, 14, 24], labels=["morning", "midday", "evening"])
mix_rate = d.groupby(["slot_bucket", "source"], observed=True)["no_show"].mean().rename("network_rate_for_mix").reset_index()
d2 = d.merge(mix_rate, on=["slot_bucket", "source"], how="left", validate="many_to_one")
adj = d2.groupby("clinic_id", observed=True).agg(rate=("no_show", "mean"), expected_from_mix=("network_rate_for_mix", "mean"), bookings=("no_show", "size"))
adj["excess_after_mix_pts"] = (adj["rate"] - adj["expected_from_mix"]) * 100
adj.sort_values("excess_after_mix_pts", ascending=False).head(5).round(3)

#### 📘 Reading the output (recomputed)

| Clinic | Rate | Expected from its mix | Bookings | Excess after mix (pts) |
|---|---|---|---|---|
| PUN-04 | 0.172 | 0.163 | 4,573 | +0.88 |
| HYD-03 | 0.171 | 0.163 | 7,072 | +0.73 |
| AHM-05 | 0.170 | 0.164 | 4,779 | +0.60 |
| AHM-02 | 0.169 | 0.163 | 9,558 | +0.59 |
| BLR-02 | 0.167 | 0.163 | 11,236 | +0.39 |
| … **CHE-03** | 0.190 | **0.192** | 7,411 | **−0.18** |

**This is the key result for brief D.** CHE-03 drops out of the top completely. About half its bookings are **evening** slots (49.6% vs 25.8% network average) and 44.9% are **phone** bookings (vs 30.3%), and those are the highest-risk combinations: network no-show is roughly 26% for evening phone bookings against roughly 12% for morning app bookings. **Given its mix, CHE-03 is exactly as expected.** Its problem is *when and how* people book there, not how the clinic is run. The action is to change the slot or source mix (reminders for evening phone bookings), not to blame the clinic manager.

> **Interview angle.** Why is drawing without replacement the same as shuffling labels here, and when would you have to shuffle for real?
>
> *What a good answer covers:* With a 0/1 outcome and one group of fixed size, the shuffled rate depends only on how many of the group's bookings are no-shows, which is a hypergeometric count. When the statistic is not a count of a binary outcome in one group, or when there are several groups compared at once, the closed form no longer applies and you permute the labels and recompute the statistic.

#### ✅ Answer: why is drawing without replacement the same as shuffling labels here, and when would you have to shuffle for real?

**Why it's the same:** shuffling clinic labels across all bookings and then looking at "this clinic" is equivalent to picking a **random subset of size n** (the clinic's number of bookings) from all bookings without replacement. The statistic, the clinic's no-show rate, depends **only on how many of those n are no-shows**, because the outcome is 0/1. The number of 1s in a random n-subset of a population with K ones out of N is, by definition, **hypergeometric(K, N−K, n)**. So instead of shuffling ~230,000 labels and counting, you draw that count directly. It's the same distribution, exact rather than approximate, and thousands of times faster.

**When you'd have to shuffle for real:**
1. **The statistic isn't a count of a binary outcome**: a mean waiting time, a median revenue, a ratio of two quantities. The distribution of a subset mean of continuous values has no simple closed form.
2. **Several groups compared at once**: a test of whether *any* of the 31 clinics differs (a max over clinics, or a chi-square-like spread). Shuffling assigns all clinics simultaneously, and their draws are **dependent** (they share the same pool), so independent hypergeometric draws per clinic would be wrong.
3. **Stratified or restricted shuffles**: permuting only within slot bucket × source (to respect mix), within city, or keeping a patient's bookings together.
4. **Paired or before/after designs**, where you permute within pairs.
5. **Statistics that depend on which specific rows** are in the group (for example the share of unique patients).

> **Interview angle.** What does your shuffle test assume, and what does the p-value it produces mean and not mean?
>
> *What a good answer covers:* It assumes bookings are exchangeable across clinics under the null, which fails if clinics differ in mix; hence the like-with-like step. The p-value is the share of shuffled worlds at least as extreme as the real one; it is not the probability the clinic is fine, and a small p on a tiny difference is not a reason to act.

#### ✅ Answer: what does the shuffle test assume, and what does its p-value mean and not mean?

**Assumptions:**
1. **Exchangeability under the null**: if the clinic made no difference, any booking could equally well have been at any clinic. That **fails when clinics differ in mix** (slot times, booking source, patient age, city). That's exactly what happened with CHE-03, and why the like-with-like adjustment follows.
2. **Independent bookings**: a patient with repeated no-shows contributes several correlated outcomes, which makes the chance band too narrow.
3. **A fixed definition of the outcome and denominator** (here cancellations count as non-no-shows).
4. **The clinic was not chosen *because* it looked worst.** Testing the **maximum** of 31 clinics against a single-clinic null overstates significance (the multiple-comparisons problem). The fair null for "the worst clinic" is the distribution of the *worst* of 31 shuffled clinics.

**What the p-value means:** the share of shuffled worlds (where the clinic label is irrelevant) in which a clinic of this size shows a rate **at least as extreme** as the one observed. Here 0 of 20,000, so p < 0.00005.

**What it doesn't mean:**
- **Not** the probability that the clinic is fine, or that the null is true.
- **Not** a measure of **how big** the difference is or whether it matters. With ~7,400 bookings, even a 0.5-point difference can produce a tiny p. Report the effect size (+2.6 pts, then −0.18 pts after mix) beside it.
- **Not** evidence of a **cause**: a small p says "not chance", but the cause might be mix, not management, as the adjustment showed.
- **Not** a reason to act on its own: a small p on a tiny difference isn't worth an intervention. Weigh the cost of acting against the size of the gap.

## 5 · The fair comparison (brief E)

Reproduce marketing's number, then look at who got the offer, then split. The grid: offer or not, before or after, by segment. A lift that holds in every segment is a lift; one that holds only in the total is a mix shift.

#### 📘 Explanation: the fair comparison

1. **Reproduce** marketing's number exactly, so you're arguing about the same thing.
2. **Look at who got the offer.** If offered and non-offered bookings differ (new vs existing patients, home vs clinic), a raw comparison mixes the offer's effect with the difference between groups (**selection**).
3. **Split** the comparison: offer or not × before or after × segment. A lift present **in every segment** is real. A lift present **only in the total** comes from a change in the mix of segments (Simpson's paradox).

#### 📘 Explanation: reproducing marketing's number and who got the offer

- `camp`: bookings tagged with a campaign. `.str.strip()` removes trailing spaces in ids, which would otherwise break the join.
- `bk.merge(camp[[...]], on="booking_id", how="left", validate="one_to_one")` attaches `campaign_id`. `offer = campaign_id.notna()`.
- `all_b`: *all* clean bookings (all quarters, not just the two halves), with a `quarter` column. `qq` = bookings per quarter.
- `r(a, b)`: % change between two quarters, or "period missing" if one doesn't exist.
- **Marketing's comparison**: Q1 2026 (campaign quarter) vs Q4 2025 (the quarter before). Then a reference comparison of Q1 2025 vs Q2 2025.
- `first_seen = all_b.groupby("person_id")["slot_at"].min()` gives each person's first booking in the pack. `is_new_patient` = first seen on or after 1 Jan 2026. Defined from history **before** the offer, so the offer can't change who counts as "new".
- `pd.crosstab(offer, is_new_patient, normalize="index")`: for offered and non-offered H1 2026 bookings, the share from new vs existing patients (each row sums to 1).

In [ ]:
camp = pd.read_csv(CONFIG["raw"] / "campaigns" / "campaign_bookings.csv.gz", dtype="string")
camp["booking_id"] = camp["booking_id"].str.strip()
bkq = bk.merge(camp[["booking_id", "campaign_id"]], on="booking_id", how="left", validate="one_to_one")
bkq["offer"] = bkq["campaign_id"].notna()
all_b = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
all_b["quarter"] = all_b["slot_at"].dt.to_period("Q").astype(str)
qq = all_b.groupby("quarter")["booking_id"].size()
r = lambda a, b: f"{(qq[a]/qq[b]-1)*100:+.1f}%" if a in qq and b in qq else "period missing in this pack"
print("marketing's comparison, Q1 2026 vs Q4 2025:", r("2026Q1", "2025Q4"), " | the same comparison a year earlier is not in the pack; Q1 2025 vs Q2 2025:", r("2025Q1", "2025Q2"))
# who got the offer: first-seen date per person, so 'new' is defined before the offer and cannot be changed by it
first_seen = all_b.groupby("person_id")["slot_at"].min()
bkq["is_new_patient"] = bkq["person_id"].map(first_seen) >= pd.Timestamp("2026-01-01")
pd.crosstab(bkq.loc[bkq["half"] == "H1_2026", "offer"], bkq.loc[bkq["half"] == "H1_2026", "is_new_patient"], normalize="index").round(3)

#### 📘 Reading the output (recomputed)

- **Marketing's comparison, Q1 2026 vs Q4 2025: +11.0%** (99,021 vs 89,196 bookings).
- Reference: Q1 2025 vs Q2 2025: +2.5%. More telling, in 2025 **Q1 was the strongest quarter (95,572) and Q4 the weakest (89,196)**. Q4 is a seasonal low, so comparing Q1 with the preceding Q4 bakes in a seasonal rise of roughly 7%. **Q1 2026 vs Q1 2025, like for like: +3.6%**, roughly the underlying trend.

| H1 2026 bookings | existing patients | new patients |
|---|---|---|
| no offer | 54.3% | 45.7% |
| **offer** | **67.4%** | **32.6%** |

The offer went disproportionately to **existing** patients, people who were already booking. And a check worth doing: **every one of the 48,582 offered bookings is a home collection, and every home collection in Q1 2026 had the offer** (the campaign is "free home collection"). So "offer vs no offer" is the same as "home vs clinic", and the two can't be separated within Q1.

Caveat: "new" means not seen since January 2025 (the pack's start), so someone last seen in 2024 counts as "new". `signup_date` in the patient master is a better definition.

#### 📘 Explanation: the split by city

`seg` = bookings per city per quarter. `pct_change(seg, "2026Q1", "2025Q4", ...)` repeats marketing's comparison **inside each city**. If every city rises by about the same amount, the cause is network-wide (season or campaign). The printed hint says to split by **channel** next.

In [ ]:
seg = all_b.groupby(["quarter", "city"])["booking_id"].size().unstack("quarter")
pct_change(seg, "2026Q1", "2025Q4", "Q1 2026 vs Q4 2025 by city, %")
print("\nIf every city shows about the same lift with and without the offer, the lift is the season, not the offer. Split by channel next.")

#### 📘 Reading the output (recomputed)

**Q1 2026 vs Q4 2025 by city:** Ahmedabad +15.0%, Bengaluru +18.7%, Hyderabad +14.9%, Pune +14.3%, Chennai +1.6%, **Jaipur −5.1%**. The campaign ran everywhere, but the lift doesn't appear in Chennai or Jaipur, so "the campaign raised bookings 11%" doesn't hold in every segment.

**The channel split the cell recommends** (bookings per quarter):

| | Q4 2025 | **Q1 2026 (offer)** | Q2 2026 (after) | Q2 2025 |
|---|---|---|---|---|
| home | 30,330 | **48,582 (+60%)** | 38,199 | 31,708 |
| clinic | 58,866 | **50,439 (−14%)** | 56,800 | 61,523 |
| total | 89,196 | 99,021 (+11%) | 94,999 | 93,231 |

Home bookings jumped by 18,252, but **clinic bookings fell by 8,427** at the same time. About half the "new" home bookings were **existing clinic patients switching channel** to get free collection (cannibalisation). Net extra bookings: about 9,800, and a seasonal Q4 → Q1 rise would have produced a large part of that anyway.

> **Interview angle.** Marketing shows bookings up 9 percent in the campaign quarter. Give three reasons that is not proof the campaign worked, and the comparison that would be.
>
> *What a good answer covers:* Seasonality (the same quarter rises every year), selection (the offer reached people who were already going to book), and mix (new patients grew for other reasons). The comparison that would show cause is like-for-like patients who did not get the offer in the same period: a holdout, or at least a matched before-behaviour comparison.

#### ✅ Answer: marketing shows bookings up 9% in the campaign quarter. Three reasons that isn't proof the campaign worked, and the comparison that would be

**Three reasons (with this pack's evidence):**
1. **Seasonality.** Q1 is seasonally strong. In 2025 it was the highest quarter and Q4 the lowest. Q1 vs the preceding Q4 rises every year, campaign or not. The like-for-like comparison, Q1 2026 vs Q1 2025, is only **+3.6%**.
2. **Selection.** The offer reached people who were going to book anyway: 67.4% of offered bookings were **existing** patients, against 54.3% of non-offered ones. Their bookings would have happened without the offer.
3. **Mix and cannibalisation.** Home bookings rose 60%, but clinic bookings fell 14% in the same quarter. Much of the "lift" is patients **moving from clinic to home** to get free collection, the same visits in a different channel, and now costing the ₹150 waiver each. On top of that, two cities (Chennai and Jaipur) didn't lift at all.

**The comparison that would show cause:**
- Best: a **randomised holdout**. Before launch, randomly withhold the offer from, say, 10% of eligible patients, then compare booking rates of offered vs holdout patients **in the same period**. Randomisation removes season, selection and mix in one step.
- Without a holdout: **difference-in-differences** on comparable groups. For example, compare the change Q4 → Q1 for home bookings this year with the same change last year (Q4 2024 → Q1 2025, if obtainable), or offered vs similar non-offered patients matched on prior booking history, city and channel. Measure **total bookings per patient** (home + clinic), not home bookings alone, so cannibalisation is netted out.

## 6 · SUM: the chart for slide 4

One chart, the claim's number against its comparison, with the denominator in the axis label. Save it to `slides/`.

#### 📘 Explanation: the chart for slide 4

- `matplotlib.use("Agg")`: a non-interactive backend that draws straight to a file. It works on a server or Codespace with no display. In Jupyter it also means the chart is **not** shown inline: open `slides/slide4_chart.png` to see it.
- `fig, ax = plt.subplots(figsize=(8, 4.2))`: one chart, 8 × 4.2 inches.
- `g = city_growth["tests"].sort_values()`: test growth per city from §1, smallest first (so the largest bar is at the top of a horizontal chart).
- `ax.barh(...)` draws horizontal bars, **pink for negative, purple for positive**: a colour that carries meaning.
- `ax.axvline(0, ...)`: a zero line, so growth and decline are read against it.
- `ax.set_xlabel(...)` puts **the denominator in the axis label**, a project rule.
- The loop `ax.text(...)` writes the value (`+7.6%`) at the end of each bar, left or right depending on sign.
- `fig.savefig(ROOT / "slides" / "slide4_chart.png", dpi=200)`: a high-resolution PNG for the slide.
- `checks.summary()` lists the checks of this notebook.

⚠️ **The axis label says "tests on collected and no-show bookings", but `bk` was never filtered by status**: it includes cancelled bookings too. Either filter `bk[bk["status"].isin(["collected","no_show"])]` before the tree, or change the label to "tests on all bookings after the identity rule". A label that doesn't match the computation is exactly what a panel catches.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
g = city_growth["tests"].sort_values()
ax.barh(g.index, g.values, color=["#D63A6A" if v < 0 else "#4E38AB" for v in g.values])
ax.axvline(0, color="#1A1440", lw=1)
ax.set_xlabel("Tests, H1 2026 vs H1 2025, percent change (denominator: tests on collected and no-show bookings, H1 2025)")
ax.set_title("Where the growth went, by city")
for i, v in enumerate(g.values): ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout()
(ROOT / "slides").mkdir(exist_ok=True)
fig.savefig(ROOT / "slides" / "slide4_chart.png", dpi=200)
print("saved slides/slide4_chart.png")
checks.summary()

#### 📘 Reading the output (recomputed)
The saved chart shows: Jaipur **−10.6%**, Chennai **+0.9%**, Hyderabad +7.2%, Bengaluru +7.6%, Pune +8.3%, Ahmedabad **+10.5%**. The checks summary lists 2 checks, both PASS: the log decomposition and the bridge balance.

## Before you move on

- Which number is on slide 3, and what are its denominator and period? Say them aloud.
- Which of the five methods did you use, and why not the nearest other one? That is an approach-log row.
- What would make you wrong? That is slide 5.

#### ✅ Answers: "Before you move on"

**1. Which number is on slide 3, and what are its denominator and period? Say them aloud.**

The `CONFIG["brief"]` letter is still a placeholder, so here's the answer for each brief, from this data:

| Brief | Slide-3 number | Numerator / denominator | Period |
|---|---|---|---|
| **A** (growth) | **Tests +5.0%**, against a plan of +18.9% (11.7% below plan) | 455,383 tests ÷ 433,611 tests, bookings after the identity rule, six cities | H1 2026 (1 Jan – 30 Jun) vs H1 2025 |
| B (Chennai/Jaipur drop) | Bookings **Chennai −4.7%, Jaipur −7.9%** year on year | Q2 2026 bookings ÷ Q2 2025 bookings, after the identity rule | Q2 2026 vs Q2 2025 |
| C (collections) | **89.8%** of invoiced collected in the quarter | ₹119.29 M ÷ ₹132.85 M invoiced (79,246 invoices) | invoices dated Q2 2026, paid by 30 Jun 2026 |
| D (clinic no-shows) | CHE-03 **19.0%** vs 16.4% network, **−0.2 pts after mix** | no-shows ÷ 7,411 clinic bookings | H1 2025 + H1 2026 |
| E (campaign) | **+3.6%** like for like, not marketing's +11% | Q1 2026 bookings ÷ Q1 2025 bookings | Q1 2026 vs Q1 2025 |

*Said aloud for brief A:* "Tests grew five percent, from 433,611 to 455,383, comparing January–June 2026 with January–June 2025 across all six cities, against a plan of nineteen percent."

**2. Which method did you use, and why not the nearest alternative?** (approach-log row 7)

For brief A, **the tree**. The question is "what is the 5% made of", which is a decomposition of a product into factors, and log growth splits it exactly (people +3.7 log-pts, visits per person −1.0, tests per visit +2.2). The nearest alternative is **the ladder**, which is built to *confirm and locate a drop* by changing one dimension per rung. It's what the city breakdown points to next (Jaipur and Chennai). We'd use it as a follow-up, but on its own it doesn't attribute growth to its components. The **bridge** works for additive flows (money in and out), not for multiplicative factors.

**3. What would make you wrong?** (slide 5)

1. **The period boundary**: `between` excludes 30 June, and the full-day figures are 455,383 vs 433,611. Use those.
2. **The identity rule**: if a nova reschedule should count as two bookings, growth is +5.8%, not +5.0%. The rule moves the headline by about 0.8 points.
3. **The system change in Chennai and Jaipur**: no overlap month to calibrate nova against legacy, and the reschedule–collection mismatch (1,449 visits) makes nova look worse on any collected-based metric.
4. **The 3,107 rejected legacy bookings**: probably real visits. They're spread evenly, so growth moves only about 0.06 points, but they're missing from levels.
5. **Patient de-duplication**: a looser or stricter rule moves the *people* branch, and with it the "more people" story.
6. **The denominator** (all bookings vs collected and no-show), which must match the chart label.